In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('..'))

import numpy as np  # noqa: F401
import pandas as pd
from statsbombpy import sb  # noqa: F401

from pipeline import get_team_matchids, get_teamseason_matchevents


In [ ]:
arsenal_1516_matchids = get_team_matchids(2, 27, 1)
display(arsenal_1516_matchids)

In [ ]:
arsenal_1516_events = get_teamseason_matchevents(2, 27, 1)

In [ ]:
event_types = arsenal_1516_events['type'].unique()

In [ ]:
print(event_types)

In [ ]:
ipevents = ['Pass', 'Ball Receipt*', 'Carry', 'Dribble', 'Shot', 'Dispossessed', 'Offside', 'Miscontrol']
arsenal_ip_events = arsenal_1516_events[arsenal_1516_events['type'].isin(ipevents)]

In [ ]:
unique_players = (
    arsenal_1516_events[["player", "player_id"]]
    .dropna()
    .drop_duplicates()
    .set_index("player_id")["player"]
    .to_dict()
)

In [ ]:
len(unique_players)

In [ ]:
print(unique_players)

In [ ]:
unique_player_matches = list(
    arsenal_1516_events[["match_id", "player_id"]]
    .dropna()
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

In [ ]:
len(unique_player_matches)

In [ ]:
pass_events = arsenal_1516_events[arsenal_1516_events['type'] == 'Pass'].dropna(axis = 1, how = 'all')

In [ ]:
display(pass_events)

In [ ]:
carry_events = arsenal_1516_events[arsenal_1516_events['type'].isin(['Carry', 'Dribble'])].dropna(axis = 1, how = 'all')

In [ ]:
display(carry_events)

Clear that carry & dribble events chain. Need to make them all the same event when they do chain

In [ ]:
dribble_events = arsenal_1516_events[arsenal_1516_events['type'] == 'Dribble'].dropna(axis = 1, how = 'all')

In [ ]:
display(dribble_events)

In [ ]:
ballreceipt_events = arsenal_1516_events[arsenal_1516_events['type'].isin(['Ball Receipt*', 'Miscontrol', 'Offside'])].dropna(axis = 1, how = 'all')

In [ ]:
display(ballreceipt_events)

In [ ]:
br_duplicates = ballreceipt_events.duplicated(subset = ['match_id', 'match_time'], keep = False)

In [ ]:
display(br_duplicates)

miscontrols aren't necessary to consider, add unnecessary noise (say nothing on player intention)

In [ ]:
broffside_events = ballreceipt_events[ballreceipt_events['type'].isin(['Ball Receipt*', 'Offside'])]

In [ ]:
broffside_duplicates = broffside_events.duplicated(subset = ['match_id', 'match_time'], keep = False)

In [ ]:
broffside_duplicates.unique()

offside events are not double counted, include

In [ ]:
defensive_br_events = arsenal_1516_events[arsenal_1516_events['type'].isin(['Interception', '50/50', 'Ball Recovery', 'Ball Receipt*'])].dropna(axis = 1, how = 'all')

3385.0 - Alexis Sanchez

from mplsoccer import Pitch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

player_events = arsenal_ip_events[
    arsenal_ip_events["player_id"].eq(3385.0)
].copy()

# Extract StatsBomb x/y coordinates from the location column
player_events[["x", "y"]] = pd.DataFrame(
    player_events["location"].tolist(),
    index=player_events.index
)

player_events = player_events.dropna(subset=["x", "y"])

event_types = sorted(player_events["type"].dropna().unique())

markers = ["o", "s", "^", "D", "P", "X", "v", "<", ">", "*", "h", "p"]
colors = plt.cm.tab20.colors

event_style = {
    event_type: {
        "marker": markers[index % len(markers)],
        "color": colors[index % len(colors)]
    }
    for index, event_type in enumerate(event_types)
}

match_ids = list(arsenal_1516_matchids)
columns = 4
rows = (len(match_ids) + columns - 1) // columns

fig, axes = plt.subplots(
    rows,
    columns,
    figsize=(18, rows * 5),
    squeeze=False
)

pitch = Pitch(
    pitch_type="statsbomb",
    pitch_color="#f7f4ed",
    line_color="#333333",
    linewidth=1
)

for index, match_id in enumerate(match_ids):
    ax = axes.flat[index]
    pitch.draw(ax=ax)

    match_events = player_events[
        player_events["match_id"].eq(match_id)
    ]

    for event_type in event_types:
        events = match_events[
            match_events["type"].eq(event_type)
        ]

        if events.empty:
            continue

        style = event_style[event_type]

        pitch.scatter(
            events["x"],
            events["y"],
            ax=ax,
            marker=style["marker"],
            s=90,
            color=style["color"],
            edgecolors="black",
            linewidth=0.5,
            alpha=0.9
        )

    ax.set_title(f"Match {match_id}", fontsize=10)

# Hide unused subplot axes
for ax in axes.flat[len(match_ids):]:
    ax.remove()

legend_handles = [
    Line2D(
        [0],
        [0],
        marker=event_style[event_type]["marker"],
        color="w",
        label=event_type,
        markerfacecolor=event_style[event_type]["color"],
        markeredgecolor="black",
        markersize=8
    )
    for event_type in event_types
]

fig.legend(
    handles=legend_handles,
    title="Event type",
    loc="center left",
    bbox_to_anchor=(0.91, 0.5)
)

fig.suptitle(
    "Player 3385.0 Events Across Arsenal Matches",
    fontsize=16,
    y=0.995
)

plt.tight_layout(rect=[0, 0, 0.9, 0.98])
plt.show()